# Titanic Survival Prediction — Decision Tree
This notebook follows the five project steps: load data, clean/encode, split, train, visualize/evaluate.

## 1. Load the Titanic dataset
Place Kaggle `train.csv` in `data/`. If it is absent, the notebook uses Seaborn's public Titanic dataset (internet required for first download).

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
csv_path = ROOT / 'data' / 'train.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path).rename(columns={
        'Pclass':'pclass', 'Age':'age', 'SibSp':'sibsp', 'Parch':'parch',
        'Fare':'fare', 'Sex':'sex', 'Embarked':'embarked', 'Survived':'survived'
    })
else:
    df = sns.load_dataset('titanic')
df.head()

## 2. Handle missing values and encode categories
A pipeline imputes numeric missing values with the median, categorical missing values with the most frequent value, and one-hot encodes categories.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric_features = ['pclass', 'age', 'sibsp', 'parch', 'fare']
categorical_features = ['sex', 'embarked']
features = numeric_features + categorical_features
df = df[features + ['survived']].dropna(subset=['survived']).copy()
df['survived'] = df['survived'].astype(int)
X = df[features]
y = df['survived']

numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median'))])
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipe, numeric_features),
    ('categorical', categorical_pipe, categorical_features)
])

## 3. Split the data
Use 80% for training and 20% for testing. Stratification keeps the survival-class proportions similar in both sets.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))

## 4. Train a DecisionTreeClassifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(
        criterion='gini', max_depth=4, min_samples_split=10,
        min_samples_leaf=5, random_state=42
    ))
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)

## 5. Visualize the tree and evaluate accuracy

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

print('Accuracy:', round(accuracy_score(y_test, predictions), 3))
print(classification_report(y_test, predictions, zero_division=0))

names = model.named_steps['preprocessor'].get_feature_names_out()
plt.figure(figsize=(22, 12))
plot_tree(model.named_steps['classifier'], feature_names=names,
          class_names=['Did not survive', 'Survived'], filled=True,
          rounded=True, fontsize=7)
plt.title('Decision Tree — Titanic Survival')
plt.show()

ConfusionMatrixDisplay.from_predictions(
    y_test, predictions, display_labels=['Did not survive', 'Survived'], cmap='Blues'
)
plt.title('Confusion Matrix')
plt.show()

## Conclusion
The decision tree learns patterns from passenger class, sex, age, family counts, fare, and embarkation port. Test accuracy and the classification report summarize performance on unseen test rows.